# Store Sales
## Feature Engineering
* **Goal:** Prepare and transform the dataset for property price prediction

This notebook prepares the cleaned dataset for machine learning models. 

In [1]:
# Import libraries

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# Define path

data_path = Path('../data')

# Find CSV
files = list(data_path.glob('*.csv'))

# Load files into dictonary
dataframes = {}

for file in files:
    name = file.stem
    dataframes[name] = pd.read_csv(file)

In [3]:
# Convert date cols to datatime format

for name, df in dataframes.items():
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])

In [4]:
# Display the shape of each dataset

for name, df in dataframes.items():
    print(f'{name}: {df.shape}')

test: (28512, 5)
train: (3000888, 6)
transactions: (83488, 3)
oil: (1218, 2)
holidays_events: (350, 6)
sample_submission: (28512, 2)
stores: (54, 5)


In [5]:
# Create a copy of the train data 
train = dataframes['train'].copy()

train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [6]:
# Display shape, cols and missing values
print('Shape:', train.shape)
print(train.columns.tolist())
print(train.isna().sum())


Shape: (3000888, 6)
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64


In [7]:
# Add Store information to train dataset
train = train.merge(
    dataframes['stores'], 
    on='store_nbr', 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}')
print(train.head())

Shape: (3000888, 10)
   id       date  store_nbr      family  sales  onpromotion   city      state  \
0   0 2013-01-01          1  AUTOMOTIVE    0.0            0  Quito  Pichincha   
1   1 2013-01-01          1   BABY CARE    0.0            0  Quito  Pichincha   
2   2 2013-01-01          1      BEAUTY    0.0            0  Quito  Pichincha   
3   3 2013-01-01          1   BEVERAGES    0.0            0  Quito  Pichincha   
4   4 2013-01-01          1       BOOKS    0.0            0  Quito  Pichincha   

  type  cluster  
0    D       13  
1    D       13  
2    D       13  
3    D       13  
4    D       13  


In [8]:
# Add transaction data
train = train.merge(
    dataframes['transactions'], 
    on=['date', 'store_nbr'], 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}')

# Check missing values for transaction
print(f'Missing transactions: {train['transactions'].isna().sum()}')

Shape: (3000888, 11)
Missing transactions: 245784


In [9]:
# Add oil price datas
train = train.merge(
    dataframes['oil'], 
    on='date', 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}]')
print(f'Missing oil prices: {train['dcoilwtico'].isna().sum()}')

Shape: (3000888, 12)]
Missing oil prices: 928422


In [10]:
# Extract time-based features from date cols
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day
train['day_of_week'] = train['date'].dt.dayofweek

# Display new features
train[['date', 'year', 'month', 'day', 'day_of_week']].head()

,date,year,month,day,day_of_week
0,2013-01-01,2013,1,1,1
1,2013-01-01,2013,1,1,1
2,2013-01-01,2013,1,1,1
3,2013-01-01,2013,1,1,1
4,2013-01-01,2013,1,1,1


In [11]:
# Identify weekends
train['is_weekend'] = train['day_of_week'].isin([5, 6]).astype(int)

# Extract quarter
train['quarter'] = train['date'].dt.quarter

# Display new featrues
train[['date', 'day_of_week', 'is_weekend', 'quarter']].head()

,date,day_of_week,is_weekend,quarter
0,2013-01-01,1,0,1
1,2013-01-01,1,0,1
2,2013-01-01,1,0,1
3,2013-01-01,1,0,1
4,2013-01-01,1,0,1


In [12]:
# Get unique dates from holiday and events dataset
special_dates = dataframes['holidays_events']['date'].unique()

# Identiy special dates
train['is_special_date'] = train['date'].isin(special_dates).astype(int)

# Check new features
print(train['is_special_date'].value_counts())

is_special_date
0    2551824
1     449064
Name: count, dtype: int64


In [13]:
# Check missing values
missing_values = train.isna().sum()

missing_percentage = (missing_values/len(train)) * 100

missing_summary = pd.DataFrame({
    'missing_values': missing_values,
    'missing_percentage': missing_percentage
})

missing_summary = missing_summary[missing_summary['missing_values'] > 0]
missing_summary

,missing_values,missing_percentage
transactions,245784,8.190376
dcoilwtico,928422,30.938242


In [14]:
# Sort by date
train = train.sort_values('date')

train['dcoilwtico'] = train['dcoilwtico'].ffill()

# Check remaining missing values
print(f'Missing oil prices: {train['dcoilwtico'].isna().sum()}')

Missing oil prices: 1782


In [15]:
# Show missing oil prices
missing_oil = train[train['dcoilwtico'].isna()]

# Check date range
print(f'First missing date: {missing_oil['date'].min()}')
print(f'Last missing date: {missing_oil['date'].max()}')
print(f'Number of missing rows: {len(missing_oil)}')

First missing date: 2013-01-01 00:00:00
Last missing date: 2013-01-01 00:00:00
Number of missing rows: 1782


In [16]:
# Fill remaining missing oil prices
train['dcoilwtico'] = train['dcoilwtico'].bfill()

# Check result
print(f'Missing oil prices: {train['dcoilwtico'].isna().sum()}')

Missing oil prices: 0


### Handling Missing Oil Prices

The oil price data contained missing values, including missing values at the beginning of the time series.

First, forward fill (`ffill()`) was used to fill missing values with the last available oil price. The remaining missing values occurred on January 1, 2013, when no previous oil price was available.

Backward fill (`bfill()`) was then applied to fill the remaining missing values using the next available oil price.

After this process, no missing values remained in the `dcoilwtico` column.

**Note:** Backward fill uses a future observation to fill a historical value. This decision should be considered when evaluating the model to avoid potential data leakage.

In [17]:
# Check oil prices
print(train['dcoilwtico'].isna().sum())
print(train['dcoilwtico'].describe())

0
count    3.000888e+06
mean     6.792490e+01
std      2.566913e+01
min      2.619000e+01
25%      4.637750e+01
50%      5.341000e+01
75%      9.572000e+01
max      1.106200e+02
Name: dcoilwtico, dtype: float64


In [23]:
# Count missing transaction values
print(f'Missing transactions: {train['transactions'].isna().sum()}')

# Calcualte percentage
print(f'Missing percentage: {train['transactions'].isna().mean()*100:.2f} %')

Missing transactions: 245784
Missing percentage: 8.19 %


In [25]:
# Select missing transactions
missing_transactions = train[train['transactions'].isna()]

# Check date range
print(f'First missing date: {missing_transactions['date'].min()}')
print(f'Last missing date: {missing_transactions['date'].max()}')

# Count unique dates and store
print(f'Unique dates: {missing_transactions['date'].nunique()}')
print(f'Unique Stores: {missing_transactions['store_nbr'].nunique()}')

First missing date: 2013-01-01 00:00:00
Last missing date: 2017-04-19 00:00:00
Unique dates: 1566
Unique Stores: 54


In [26]:
# Count stores with missing transactions for each date
missing_stores_per_day = (
    missing_transactions.groupby('date')['store_nbr'].nunique()
)

# Show distribution
print(missing_stores_per_day.value_counts().sort_index())

store_nbr
1     440
2      91
3      97
4      65
5      97
6     164
7     406
8     197
11      1
18      1
40      1
52      1
53      3
54      2
Name: count, dtype: int64


In [27]:
# Keep only unique date-store combinations with missing transactions
missing_pairs = (
    train.loc[train['transactions'].isna(), ['date', 'store_nbr']]
    .drop_duplicates()
)

# Count missing days for each store
missing_days_per_store = (
    missing_pairs.groupby('store_nbr')['date']
    .nunique()
    .sort_values()
)

print(missing_days_per_store)

store_nbr
26       6
31       6
33       6
34       6
37       6
38       6
39       6
23       7
32       7
16       7
28       7
27       7
44       7
45       7
46       7
47       7
48       7
49       7
5        7
50       7
51       7
2        7
41       7
40       7
35       8
1        8
13       8
3        8
4        8
6        8
8        8
9        8
11       8
15       8
54       8
19       8
10       9
7        9
17      10
43      12
30      29
14      46
12      68
25      69
24     107
18     118
36     133
53     517
20     775
29     810
21     936
42     964
22    1013
52    1566
Name: date, dtype: int64


In [28]:
# Show available holiiday types
print(dataframes['holidays_events']['type'].value_counts())

# Show avalaible holidy descriptions
print(dataframes['holidays_events']['description'].head(10))

type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64
0               Fundacion de Manta
1    Provincializacion de Cotopaxi
2              Fundacion de Cuenca
3        Cantonizacion de Libertad
4        Cantonizacion de Riobamba
5           Cantonizacion del Puyo
6        Cantonizacion de Guaranda
7    Provincializacion de Imbabura
8       Cantonizacion de Latacunga
9             Fundacion de Machala
Name: description, dtype: object


In [31]:
# Show holday related cols
print(dataframes['holidays_events'][
    ['date', 'type', 'locale', 'locale_name', 'transferred']
].head(15))

         date     type    locale    locale_name  transferred
0  2012-03-02  Holiday     Local          Manta        False
1  2012-04-01  Holiday  Regional       Cotopaxi        False
2  2012-04-12  Holiday     Local         Cuenca        False
3  2012-04-14  Holiday     Local       Libertad        False
4  2012-04-21  Holiday     Local       Riobamba        False
5  2012-05-12  Holiday     Local           Puyo        False
6  2012-06-23  Holiday     Local       Guaranda        False
7  2012-06-25  Holiday  Regional       Imbabura        False
8  2012-06-25  Holiday     Local      Latacunga        False
9  2012-06-25  Holiday     Local        Machala        False
10 2012-07-03  Holiday     Local  Santo Domingo        False
11 2012-07-03  Holiday     Local      El Carmen        False
12 2012-07-23  Holiday     Local        Cayambe        False
13 2012-08-05  Holiday     Local     Esmeraldas        False
14 2012-08-10  Holiday  National        Ecuador        False


In [35]:
# Get dates of national holidays
national_holidays = dataframes['holidays_events'].loc[
    (dataframes['holidays_events']['type'] == 'Holiday') & 
    (dataframes['holidays_events']['locale'] == 'National'),
    'date'
].unique()

# Finding missing transaction on national holidays
missing_on_national_holidays = train.loc[
    train['transactions'].isna() & 
    train['date'].isin(national_holidays)
]

print(f'Missing national holidays: {len(missing_on_national_holidays)}')

Missing national holidays: 14586


In [39]:
# Get dates of local and regional holidays
local_holidays = dataframes['holidays_events'].loc[
    (dataframes['holidays_events']['type'] == 'Holiday') &
    (dataframes['holidays_events']['locale'].isin(['Local', 'Regional'])),
    ["date", "locale", "locale_name"]
].drop_duplicates()

# Find missing transactions on local or regional holiday dates
missing_on_local_holidays = train.loc[
    train['transactions'].isna() &
    train['date'].isin(local_holidays['date'])
]

print(f'Missing rows on local or regional holidays: {len(missing_on_local_holidays)}')

print(f'Unique dates: {missing_on_local_holidays['date'].nunique()}')

Missing rows on local or regional holidays: 14751
Unique dates: 101


In [40]:
# Filling missing transaction using the median of each store 
train['transactions'] = (
    train.groupby('store_nbr')['transactions'].transform(lambda x: x.fillna(x.median()))
)

# Check remaining missing values
print(f'Missing transactions: {train['transactions'].isna().sum()}')

Missing transactions: 0


### Handling Missing Transaction Values

Missing transaction values were identified in 245,784 rows (8.19% of the training data).

National, regional, and local holidays were examined to investigate potential patterns in the missing values. However, missing transactions on holidays do not necessarily indicate that stores were closed.

To keep preprocessing simple and consistent, missing transaction values were replaced with the median transaction count of the corresponding store.

After imputation, no missing transaction values remained.